In [ ]:
using_colab = True

if using_colab:
    import sys
    !git clone https://github.com/facebookresearch/sam3.git /content/sam3
    %cd /content/sam3
    !{sys.executable} -m pip install -e ".[notebooks]"
    !{sys.executable} -m pip install --upgrade --force-reinstall --no-cache-dir \
        "numpy==2.2.6" \
        "scipy==1.15.3" \
        "scikit-image==0.25.2" \
        "scikit-learn==1.7.2" \
        "pillow==11.3.0" \
        opencv-python matplotlib

In [ ]:
!hf auth login

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

import sam3
from PIL import Image
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import plot_results

sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")

In [ ]:
import torch

# turn on tfloat32 for Ampere GPUs
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

In [ ]:
bpe_path = os.path.join(
    os.path.dirname(sam3.__file__),
    "assets",
    "bpe_simple_vocab_16e6.txt.gz",
)
model = build_sam3_image_model(bpe_path=bpe_path)

In [ ]:
image_path = "/content/lat_-2.0_long_-16.0_original.png"
image = Image.open(image_path)
width, height = image.size
processor = Sam3Processor(model, confidence_threshold=0.5)
inference_state = processor.set_image(image)

In [ ]:
processor.reset_all_prompts(inference_state)
inference_state = processor.set_text_prompt(
    state=inference_state,
    prompt="crater",
)

img0 = Image.open(image_path)
plot_results(img0, inference_state)

In [ ]:
processor.reset_all_prompts(inference_state)
inference_state = processor.set_text_prompt(
    state=inference_state,
    prompt="circle",
)

img0 = Image.open(image_path)
plot_results(img0, inference_state)